# Stage 9a — Diagnostic : pourquoi le F1 plafonne

Deux soumissions ont donné ~0.82 alors que notre CV honnête disait ~0.93. Le score = 60% F1 + 40% AUC, et **on ne peut pas bouger le seuil de 0.5**. Ce notebook montre *exactement* où on perd des points.

In [1]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
TRAIN_PATH="Train.csv"; TEST_PATH="Test.csv"; SAMPLE_PATH="SampleSubmission.csv"
NODATA=-9999
MONTHS=[f"{i:02d}" for i in range(1,13)]
BANDS=["VH","VV","blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]
# profil de donnees manquantes MESURE sur le vrai test (Stage 8)
WINTER_BIAS=np.array([.88,.78,.62,.50,.41,.43,.36,.41,.50,.80,.75,.87])

def load_raw(p): return pd.read_csv(p).replace(NODATA, np.nan)
def safe_ratio(a,b): return (a-b)/(a+b)

def build_features(df):
    feats={}; idx={}
    for m in MONTHS:
        g=df[f"green_{m}"]; n=df[f"nir_{m}"]; r=df[f"red_{m}"]; s1=df[f"swir1_{m}"]
        idx.setdefault("NDWI",[]).append(safe_ratio(g,n))
        idx.setdefault("MNDWI",[]).append(safe_ratio(g,s1))
        idx.setdefault("NDVI",[]).append(safe_ratio(n,r))
        idx.setdefault("NDMI",[]).append(safe_ratio(n,s1))
        idx.setdefault("VHmVV",[]).append(df[f"VH_{m}"]-df[f"VV_{m}"])
    def summ(monthly,name):
        a=np.column_stack([np.asarray(x,float) for x in monthly])
        feats[f"{name}_mean"]=np.nanmean(a,1); feats[f"{name}_std"]=np.nanstd(a,1)
        feats[f"{name}_min"]=np.nanmin(a,1);  feats[f"{name}_max"]=np.nanmax(a,1)
        feats[f"{name}_range"]=np.nanmax(a,1)-np.nanmin(a,1); feats[f"{name}_med"]=np.nanmedian(a,1)
        feats[f"{name}_nvalid"]=np.isfinite(a).sum(1)
    for b in BANDS: summ([df[f"{b}_{m}"] for m in MONTHS], b)
    for k,v in idx.items(): summ(v,k)
    return pd.DataFrame(feats,index=df.index).replace([np.inf,-np.inf],np.nan)

def corrupt_realistic(df, seed=0):
    rng=np.random.RandomState(seed); d=df.copy()
    cols={b:[f"{b}_{m}" for m in MONTHS] for b in BANDS}
    arr={b:d[cols[b]].values.astype(float) for b in BANDS}
    for i in range(len(d)):
        miss=rng.rand(12)<WINTER_BIAS*rng.uniform(0.7,1.1)
        for b in BANDS: arr[b][i,miss]=np.nan
    for b in BANDS: d[cols[b]]=arr[b]
    return d

In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix

def make_hgb(**kw):
    return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
            max_leaf_nodes=7, l2_regularization=1.0, random_state=42, **kw)
def make_lr(**kw):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=2000, **kw))

def fit_ensemble(train_df, ytr, X_eval, class_weight=None, n_corrupt=2):
    """Augmente (realiste) + ensemble HGB+LR. class_weight={0:w,1:1} penalise les faux positifs."""
    parts=[train_df]+[corrupt_realistic(train_df,seed=i+1) for i in range(n_corrupt)]
    aug=pd.concat(parts,ignore_index=True); yaug=np.concatenate([ytr]*len(parts))
    Xa=build_features(aug)
    h=make_hgb(class_weight=class_weight).fit(Xa,yaug)
    l=make_lr(class_weight=class_weight).fit(Xa,yaug)
    return 0.5*h.predict_proba(X_eval)[:,1]+0.5*l.predict_proba(X_eval)[:,1]

In [3]:
train=load_raw(TRAIN_PATH); y=train["label"].values
X=build_features(train)
clusters=KMeans(10,random_state=42,n_init=10).fit_predict(X.fillna(X.median()))
gkf=GroupKFold(5)

# CV honnete (validation corrompue = vrai test) avec le modele ACTUEL (sans correction)
oof=np.zeros(len(y))
for tr,va in gkf.split(X,y,clusters):
    Xv=build_features(corrupt_realistic(train.iloc[va],seed=99))
    oof[va]=fit_ensemble(train.iloc[tr],y[tr],Xv,class_weight=None)
pred=(oof>=0.5).astype(int)

## Le diagnostic au seuil 0.5

In [4]:
print("AUC = %.3f  (excellent classement)"%roc_auc_score(y,oof))
print("F1  = %.3f  (c'est ici qu'on bloque)"%f1_score(y,pred))
print("Precision = %.3f   Rappel = %.3f"%(precision_score(y,pred),recall_score(y,pred)))
tn,fp,fn,tp=confusion_matrix(y,pred).ravel()
print(f"\nFaux positifs={fp}   Faux negatifs={fn}")
print(f"Taux predit positif = {pred.mean():.1%}   vs vrai taux = {y.mean():.1%}")
print("\n-> Le modele SUR-PREDIT les bassins: beaucoup de faux positifs,")
print("   precision basse, et il crie 'bassin' ~46% du temps au lieu de ~40%.")

AUC = 0.983  (excellent classement)
F1  = 0.893  (c'est ici qu'on bloque)
Precision = 0.837   Rappel = 0.958

Faux positifs=137   Faux negatifs=31
Taux predit positif = 46.2%   vs vrai taux = 40.4%

-> Le modele SUR-PREDIT les bassins: beaucoup de faux positifs,
   precision basse, et il crie 'bassin' ~46% du temps au lieu de ~40%.


## La preuve : si on POUVAIT bouger le seuil (interdit, juste pour comprendre)

In [5]:
print("F1 selon le seuil (DIAGNOSTIC — le concours impose 0.5):")
for t in [0.3,0.4,0.5,0.6,0.7]:
    p=(oof>=t).astype(int)
    print(f"  seuil={t}: F1={f1_score(y,p):.3f}  taux_positif={p.mean():.1%}")
print("\n-> Le F1 serait meilleur a 0.6-0.7. Comme c'est INTERDIT,")
print("   il faut rendre le modele MOINS sur-confiant pour que 0.5 devienne le bon point.")

F1 selon le seuil (DIAGNOSTIC — le concours impose 0.5):
  seuil=0.3: F1=0.879  taux_positif=49.4%
  seuil=0.4: F1=0.887  taux_positif=47.9%
  seuil=0.5: F1=0.893  taux_positif=46.2%
  seuil=0.6: F1=0.911  taux_positif=44.0%
  seuil=0.7: F1=0.918  taux_positif=41.7%

-> Le F1 serait meilleur a 0.6-0.7. Comme c'est INTERDIT,
   il faut rendre le modele MOINS sur-confiant pour que 0.5 devienne le bon point.


## Conclusion
- **AUC excellente (~0.98), F1 bridé (~0.90)** : le classement est bon, mais au seuil 0.5 on a trop de **faux positifs**.
- Cause : l'augmentation rend le modèle "penche-bassin" → probabilités gonflées.
- On ne peut pas bouger le seuil → on doit **recalibrer le modèle** (Stage 9b).